In [1]:
import json
import orjson
import msgpack
import time
from collections import defaultdict

In [2]:
LEXICON_PATH = "../Lexicon/lexicons_ids_new.json"

with open(LEXICON_PATH, "rb") as f:
    lexicon = orjson.loads(f.read())

print(f"Lexicon size: {len(lexicon)}")


Lexicon size: 310564


In [4]:
TF_PATH = "../Barrels/term_frequencies.json"

with open(TF_PATH, "rb") as f:
    tf_map = orjson.loads(f.read())

print(f"Loaded TFs for {len(tf_map)} terms")


Loaded TFs for 310564 terms


In [5]:
tfs = list(tf_map.values())

print("Min TF:", min(tfs))
print("Max TF:", max(tfs))
print("Median TF:", sorted(tfs)[len(tfs)//2])


Min TF: 4
Max TF: 80627
Median TF: 16


In [7]:
MIN_TF = 5
MIN_LEN = 4

def is_valid_autocomplete_term(word, tf):
    if tf < MIN_TF:
        return False
    if len(word) < MIN_LEN:
        return False
    if not word.isalpha():
        return False
    return True


In [8]:
autocomplete_vocab = []

for word in lexicon.keys():
    tf = tf_map.get(word, 0)
    if is_valid_autocomplete_term(word, tf):
        autocomplete_vocab.append((word, tf))

print(f"Autocomplete candidate words: {len(autocomplete_vocab)}")


Autocomplete candidate words: 249719


In [9]:
class TrieNode:
    __slots__ = ("children", "word", "tf")

    def __init__(self):
        self.children = {}
        self.word = None
        self.tf = 0


In [10]:
root = TrieNode()

def insert(word, tf):
    node = root
    for ch in word:
        node = node.children.setdefault(ch, TrieNode())
    node.word = word
    node.tf = tf

start = time.time()
for word, tf in autocomplete_vocab:
    insert(word, tf)

print(f"Trie built in {(time.time() - start):.2f} sec")


Trie built in 1.31 sec


In [11]:
def autocomplete(prefix, k=5):
    node = root
    for ch in prefix:
        if ch not in node.children:
            return []
        node = node.children[ch]

    results = []

    def dfs(n):
        if n.word:
            results.append((n.word, n.tf))
        for child in n.children.values():
            dfs(child)

    dfs(node)
    results.sort(key=lambda x: x[1], reverse=True)
    return [w for w, _ in results[:k]]


In [12]:
def split_query(query):
    parts = query.strip().split()
    if len(parts) == 1:
        return [], parts[0].lower()
    return parts[:-1], parts[-1].lower()


In [13]:
queries = [
    "data str",
    "machine lear",
    "search algo",
    "information retr"
]

for q in queries:
    context, last = split_query(q)

    start = time.time()
    completions = autocomplete(last, k=5)
    elapsed = (time.time() - start) * 1000

    suggestions = [
        " ".join(context + [w]) for w in completions
    ]

    print(f"Query: '{q}'")
    print("Suggestions:", suggestions)
    print(f"Time: {elapsed:.2f} ms\n")


Query: 'data str'
Suggestions: ['data structure', 'data strong', 'data strategy', 'data strategies', 'data strain']
Time: 0.00 ms

Query: 'machine lear'
Suggestions: ['machine learn', 'machine learning', 'machine learned', 'machine learnt', 'machine learns']
Time: 0.00 ms

Query: 'search algo'
Suggestions: ['search algorithm', 'search algorithms', 'search algorithmic', 'search algonquin', 'search algo']
Time: 0.00 ms

Query: 'information retr'
Suggestions: ['information retrieved', 'information retrospective', 'information retrovirus', 'information retroviral', 'information retroviruses']
Time: 0.00 ms

